# PathSnap - EDA (M1)
All numbers/plots below are produced by code in this repo and saved to `results/m1/`.
Partition: **train** (80 trips) unless a cell says otherwise; `student_test` (10 trips) is shown alongside for comparison.
*Add 1-3 sentences of your own observations under each section (the viva will ask).*

In [ ]:
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT/'src'), str(ROOT/'scripts')]
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from IPython.display import Image, display
import run_eda as E
DATA_ROOT = Path(os.environ.get('PATHSNAP_DATA_ROOT', ROOT/'data'/'raw'))
OUT = Path(os.environ.get('PATHSNAP_OUT', ROOT/'results'/'m1'))
SPLITS = ['train', 'student_test']
FLAG_MEDIAN_M = 50.0
pd.set_option('display.width', 200, 'display.max_columns', 60)

## 1-6. Per-trip statistics (inventory, sampling, validity, extent, network, route)
One pass over every trip (takes a few minutes on the full package). Raises `n_jobs` to use more cores.

In [ ]:
df, dts, rds = E.collect_trip_stats(SPLITS, DATA_ROOT, OUT, flag_median_m=FLAG_MEDIAN_M, n_jobs=1)
s = E.summarise(df, OUT)
E.plot_all(df, dts, rds, OUT, FLAG_MEDIAN_M)
df.shape

### 1. Inventory

In [ ]:
s['inventory'].T

### 2-3. Sampling and missing / invalid data

In [ ]:
s['quality'].T

In [ ]:
display(Image(str(OUT/'dt_hist.png')))

### 4. Geographic extent

In [ ]:
display(Image(str(OUT/'world_centroids.png')))
df.groupby('split')[['bbox_w_km','bbox_h_km']].describe().T

### 5. Road network

In [ ]:
display(Image(str(OUT/'network_size.png')))
df.groupby('split')[['weak_components','strong_components','largest_weak_share','largest_strong_share','isolated_nodes','twin_ratio','zero_len_arcs','arc_len_median_m','arc_len_p95_m']].describe().T

### 6. Route analysis and GPS -> reference-route distance (exposes offset trips)

In [ ]:
display(Image(str(OUT/'gps_to_route_boxplot_train.png')))
display(Image(str(OUT/'gaps_and_route_length.png')))
df[df.split=='train'][['route_arcs','route_km','route_breaks','route_repeated_arcs','gps_route_dist_median_m','gps_route_dist_p95_m']].describe().T

**Offset / suspicious trips found by the rule** `median GPS->route distance > 50 m` (not hand-picked) - these are excluded from training labels (`label_valid=False`):

In [ ]:
df.loc[df.flagged_offset == True, ['split','trip_id','gps_route_dist_median_m','gps_route_dist_p95_m','pct_gps_gt50m_from_route']]

## 7. Candidate generation: miss rate vs radius (train)
Miss rate = share of valid observations whose ground-truth arc is not among the candidates. Labels come from M3's alignment when `pathsnap.labels` exists, otherwise the documented fallback (see `label_source` column).

In [ ]:
cs = E.candidate_stats(DATA_ROOT, OUT, 'train', flag_median_m=FLAG_MEDIAN_M, n_jobs=1)
cs

In [ ]:
display(Image(str(OUT/'miss_rate_vs_radius.png')))

## 9. Data-quality issues, assumptions and decisions
*Fill from the tables above in your own words:*

| Issue | Evidence (from results/m1) | Decision |
|---|---|---|
| CRLF line endings | | whitespace-regex parsing |
| Sampling gaps (> 10 s) | | split into segments; windows/heading never cross a gap |
| Offset trips | | flagged by rule, `label_valid=False` |
| Route breaks / loops | | |
| Twin arcs (identical geometry) | | heading features |
| Worldwide extent | | per-trip UTM projection |